# summarizer.ipynb
## Purpose
- The purpose of this notebook is to conduct some rudimentary preprocessing work on datasets that we will be using to train the pre-trained encoder model that we will be importing.
- We will be training the model on a few different kinds of summary data, where the amount of information given will vary along with the output.

## Process
1. We will import data from Hugging Face using the datasets library.
2. Conduct some small EDA on the dataset(s)
3. Export and save the datasets to .csv files (or store them in another way.)

### Step 1 - Install and Load in Necessary Libaries
- Install datasets library.
- Import/download necessary libraries.
- Import the datasets we need (CNN/DailyMail, XSum [both recommended by CoPilot])

In [1]:
from datasets import load_dataset

cnn_daily = load_dataset("cnn_dailymail", "3.0.0")

In [2]:
xsum = load_dataset("xsum")

### Step 2 - Create Preprocessing Pipeline for both CNN/DailyMail and XSum
- Use BART-large tokenizer to tokenize the data (both the inputs and the targets).
- Use a function that parses through the datasets and ensures that the fields are the same.
- Combine the datasets to feed into the model, split them by train and validation.

In [3]:
from transformers import BartTokenizer

tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")

In [4]:
def preprocess_dataset(batch, dataset_name):
    if dataset_name == "cnn":
        inputs = batch["article"]
        targets = batch["highlights"]
    elif dataset_name == "xsum":
        inputs = batch["document"]
        targets = batch["summary"]

    model_inputs = tokenizer(inputs, max_length=1024, truncation=True, padding="max_length", return_tensors="pt")
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length", return_tensors="pt")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [5]:
cnn_tokenized = cnn_daily.map(
    lambda batch: preprocess_dataset(batch, "cnn"),
    batched=True,
    remove_columns=cnn_daily["train"].column_names
)

Map:   0%|          | 0/287113 [00:00<?, ? examples/s]

Exception in thread Thread-3:
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3312.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "C:\Users\samee\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\_monitor.py", line 84, in run
    instance.refresh(nolock=True)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "C:\Users\samee\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\std.py", line 1347, in refresh
    self.display()
    ~~~~~~~~~~~~^^
  File "C:\Users\samee\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\notebook.py", line 171, in display
    rtext.value = right
    ^^^^^^^^^^^
  File "C:\Users\samee\AppData\Local\Packages\Python

Map:   0%|          | 0/13368 [00:00<?, ? examples/s]

Map:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [6]:
xsum_tokenized = xsum.map(
    lambda batch: preprocess_dataset(batch, "xsum"),
    batched=True,
    remove_columns=xsum["train"].column_names
)

Map:   0%|          | 0/204045 [00:00<?, ? examples/s]

Map:   0%|          | 0/11332 [00:00<?, ? examples/s]

Map:   0%|          | 0/11334 [00:00<?, ? examples/s]

In [7]:
from datasets import concatenate_datasets

combined_train = concatenate_datasets([
    cnn_tokenized['train'],
    xsum_tokenized['train']
])

combined_validation = concatenate_datasets([
    cnn_tokenized['validation'],
    xsum_tokenized['validation']
])

In [8]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    combined_train,
    batch_size = 2,
    shuffle = True
)

val_loader = DataLoader(
    combined_validation,
    batch_size=2
)

### Step 3 - Load in the model and commence training
- Load in the libraries to import BART-large

In [9]:
import torch
from torch.optim import AdamW
from transformers import (
    BartForConditionalGeneration,
    get_linear_schedule_with_warmup
)
from tqdm import tqdm

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [11]:
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large")
model = model.to(device)

In [12]:
epochs = 1
batch_size = 2
learning_rate = 3e-5
warmup_steps = 500
gradient_accumulation_steps = 4  # effective batch size = 2 * 4 = 8

In [13]:
optimizer = AdamW(model.parameters(), lr=learning_rate)

total_steps = len(train_loader) * epochs // gradient_accumulation_steps

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [14]:
model.train()

for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    epoch_loss = 0

    for step, batch in enumerate(tqdm(train_loader)):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss = loss / gradient_accumulation_steps
        loss.backward()

        if (step + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        epoch_loss += loss.item()

    print(f"Epoch Loss: {epoch_loss:.4f}")

Epoch 1/1


  0%|          | 0/245579 [00:00<?, ?it/s]


AttributeError: 'list' object has no attribute 'to'